# Metadata Extraction from Europeana

This notebook belongs to **AISTER Step 1 (Text-based metadata enrichment)** and downloads structured metadata for the folk paintings and icons subset of the Krovets collection.

- Input: Europeana Set API ([set/26106](https://api.europeana.eu/set/26106.json)) — 312 folk paintings and icons.
- Output: `krovets_folk_metadata.csv` with one row per record and normalized metadata fields.

**How to run**
1. Get an API key by creating an account on [Europeana](https://www.europeana.eu/).
2. Set your API key:
   - **Colab**: open **Secrets** (key icon), add `EUROPEANA_API_KEY`, and paste your key as the value.
   - **Jupyter/local**: set an environment variable named `EUROPEANA_API_KEY`.
3. Run all cells from top to bottom.
4. If you are using Colab and need Drive access, run the Drive mount cell and authorize access when prompted.

## Setup

Install required dependencies.

In [1]:
!pip install requests pandas

Import all libraries used in this notebook.

In [2]:
import os
import re
import requests
import pandas as pd

try:
    from google.colab import drive, userdata
except ImportError:
    drive = None
    userdata = None

Mount Google Drive (Colab only). Skip this cell in Jupyter/local unless you need Colab Drive paths.

In [3]:
if drive is not None:
    drive.mount('/content/drive')
else:
    print('Not running in Colab; skipping Drive mount.')

Mounted at /content/drive


Load the Europeana API key.

In [4]:
if userdata is not None:
    EUROPEANA_API_KEY = userdata.get('EUROPEANA_API_KEY')
else:
    EUROPEANA_API_KEY = os.environ.get('EUROPEANA_API_KEY')

if not EUROPEANA_API_KEY:
    raise ValueError('EUROPEANA_API_KEY not found. Set it in Colab Secrets or as an environment variable.')

## Fetch item IDs from the collection

Folk paintings and icons are fetched from the Europeana Set API endpoint ([set/26106](https://api.europeana.eu/set/26106.json)). The set contains 312 items; we paginate through all pages to collect their Europeana IDs.

In [ ]:
url = "https://api.europeana.eu/set/26106.json"

headers = {"X-API-KEY": EUROPEANA_API_KEY}

item_ids = []
page = 1
page_size = 100

while True:
    params = {
        'profile': 'items',
        'page': page,
        'pageSize': page_size
    }

    response = requests.get(url, params=params, headers=headers, timeout=60)
    response.raise_for_status() # Raise an HTTPError for bad responses
    data = response.json()
    items = data.get('items', [])

    if not items:
        break

    # URIs look like "http://data.europeana.eu/item/1413/KYD1241"
    # The Search API needs the europeana_id in "/1413/KYD1241" format (last two segments)
    item_ids.extend(['/'.join(uri.split('/')[-2:]) for uri in items])
    print(f"Fetched {len(item_ids)} items so far (page {page})")

    if 'next' not in data:
        break
    page += 1

print(f"Successfully fetched metadata for {len(item_ids)} items.")

Fetched 100 items so far (page 1)
Fetched 200 items so far (page 2)
Fetched 300 items so far (page 3)
Fetched 312 items so far (page 4)
Successfully fetched metadata for 312 items.


In [6]:
print(item_ids[:5])

['1413/KYD1241', '1413/KYD1242', '1413/KYD1243', '1413/KYD1244', '1413/KYD1245']


## Fetch rich metadata via the Search API

Using the item IDs from the previous step, we query the Europeana Search API in batches of 100 to retrieve full (`profile=rich`) metadata for each record.

In [7]:
search_url = "https://api.europeana.eu/record/v2/search.json"
all_items = []
batch_size = 100

for i in range(0, len(item_ids), batch_size):
    batch = item_ids[i:i + batch_size]
    id_query = " OR ".join([f'europeana_id:\"/{eid}\"' for eid in batch])
    cursor = "*"
    while cursor:
        resp = requests.get(search_url, params={
            "query": id_query,
            "rows": batch_size,
            "cursor": cursor,
            "profile": "rich",
        }, headers=headers, timeout=60)
        resp.raise_for_status()
        data = resp.json()
        fetched = data.get("items", [])
        if not fetched:
            break
        all_items.extend(fetched)
        cursor = data.get("nextCursor")
    print(f"Fetched {len(all_items)} / {len(item_ids)} records")

print(f"Successfully fetched metadata for {len(all_items)} items.")

Fetched 100 / 312 records
Fetched 200 / 312 records
Fetched 300 / 312 records
Fetched 312 / 312 records
Successfully fetched metadata for 312 items.


Inspect the first record to understand the JSON structure before extracting fields.

In [8]:
all_items[0]

{'completeness': 0,
 'country': ['Ukraine'],
 'dataProvider': ['Online Museum of the traditional art of Ukraine - KROVETS'],
 'dcCreator': ['unknown'],
 'dcCreatorLangAware': {'def': ['unknown']},
 'dcDescription': ['Folk Icon "Wedding couple", Ukraine, Hutsulshchyna, Ivano-Frankivsk obl., Verkhovynskyi rn., v. Bukovets, Technique: painting, inlay on wood, Materials: wooden frame, gouache paints, paper, Creator: unknown, mid XX century'],
 'dcDescriptionLangAware': {'en': ['Folk Icon "Wedding couple", Ukraine, Hutsulshchyna, Ivano-Frankivsk obl., Verkhovynskyi rn., v. Bukovets, Technique: painting, inlay on wood, Materials: wooden frame, gouache paints, paper, Creator: unknown, mid XX century']},
 'dcSubjectLangAware': {'def': ['http://www.wikidata.org/entity/Q3305213',
   'http://www.wikidata.org/entity/Q132137',
   'http://www.wikidata.org/entity/Q135109717']},
 'dcTitleLangAware': {'en': ['Folk Icon "Wedding couple"']},
 'dcTypeLangAware': {'en': ['painting']},
 'dctermsSpatial': ['

## Extract and normalize metadata

Parse each record into a flat row. Most fields map directly to JSON keys, but a few require special handling (see comments in the code below).

In [11]:
rows = []
for item in all_items:
    desc = (item.get('dcDescription') or [''])[0]

    # Medium: not a standalone field. Parsed from dcDescription free text.
    # Descriptions follow "...Materials: X, Y, Z, Creator: ..." so we grab
    # everything between 'Materials:' and ', Creator:' (or end of string).
    medium_match = re.search(r'Materials:\s*(.*?)(?:,\s*Creator:|$)', desc)
    medium = medium_match.group(1).strip().rstrip(',') if medium_match else ''

    # Places: dctermsSpatial mixes geonames/Europeana URIs with human-readable
    # strings (e.g. 'Ivano-Frankivsk obl., Verkhovynskyi rn., v. Bukovets').
    # We discard the URIs and keep only the readable entries.
    spatial = item.get('dctermsSpatial') or []
    places = [s for s in spatial if isinstance(s, str) and not s.startswith('http')]

    # Subject: dcSubjectLangAware only has raw Wikidata URIs, not labels.
    # edmConceptPrefLabelLangAware contains the resolved English labels
    # (e.g. ['painting', 'icon', 'Hutsulshchyna - ethnographical region']).
    subjects = (item.get('edmConceptPrefLabelLangAware') or {}).get('en', [])

    # Creation date: edmTimespanLabelLangAware has English period labels
    # (e.g. '20th century'). The same label often appears twice (from two
    # timespan entities), so we deduplicate with dict.fromkeys.
    dates = list(dict.fromkeys((item.get('edmTimespanLabelLangAware') or {}).get('en', [])))

    type_d = item.get('dcTypeLangAware') or {}

    rows.append({
        'europeana_id': (item.get('id', '').split('/')[-1] if item.get('id') else ''),
        'Title': (item.get('title') or [''])[0],
        'Description': desc,
        'ImageLink': (item.get('edmIsShownBy') or [''])[0],
        'Creator': (item.get('dcCreator') or [''])[0],
        'Subject': ' | '.join(subjects),
        'Type of item': (type_d.get('en') or [''])[0],
        'Medium': medium,
        'Providing institution': (item.get('dataProvider') or [''])[0],
        'Aggregator': (item.get('provider') or [''])[0],
        'Rights statement': (item.get('rights') or [''])[0],
        'Creation date': ' | '.join(dates),
        'Places': ' | '.join(places),
        'Identifier': item.get('id', ''),
        'Is part of': (item.get('edmDatasetName') or [''])[0],
        'Providing country': (item.get('country') or [''])[0],
        'Collection name': (item.get('europeanaCollectionName') or item.get('edmDatasetName') or [''])[0],
        'First time published on Europeana': item.get('timestamp_created', ''),
        'Last time updated from providing institution': item.get('timestamp_update', ''),
    })

df = pd.DataFrame(rows)
print(f"Extracted {len(df)} rows with {len(df.columns)} columns.")

Extracted 312 rows with 19 columns.


Preview the first few rows to confirm the extraction looks correct.

In [12]:
df.head()

,europeana_id,Title,Description,ImageLink,Creator,Subject,Type of item,Medium,Providing institution,Aggregator,Rights statement,Creation date,Places,Identifier,Is part of,Providing country,Collection name,First time published on Europeana,Last time updated from providing institution
0,HMT1,"Folk Icon ""Wedding couple""","Folk Icon ""Wedding couple"", Ukraine, Hutsulshc...",https://krovets.ua/disk/products/QCbv4ZU6ojKZH...,unknown,painting | icon | Paper | Hutsulshchyna - ethn...,painting,"wooden frame, gouache paints, paper",Online Museum of the traditional art of Ukrain...,MUSEU,http://creativecommons.org/licenses/by-nc-nd/4.0/,20th century,"Ivano-Frankivsk obl., Verkhovynskyi rn., v. Bu...",/1413/HMT1,1413_KROVETS_Museum,Ukraine,1413_KROVETS_Museum,2025-12-05T16:01:05.765Z,2025-12-05T16:01:05.765Z
1,HMT2,"Folk Icon ""St. Michael""","Folk Icon ""St. Michael"", Ukraine, Hutsulshchyn...",https://krovets.ua/disk/products/JJNPuDEex6R6W...,unknown,Wood | painting | Canvas | icon | Hutsulshchyn...,printing,"clock face, wood, levkas, canvas",Online Museum of the traditional art of Ukrain...,MUSEU,http://creativecommons.org/licenses/by-nc-nd/4.0/,20th century,"Ivano-Frankivsk obl., Verkhovynskyi rn., v. Bu...",/1413/HMT2,1413_KROVETS_Museum,Ukraine,1413_KROVETS_Museum,2025-12-05T16:01:05.765Z,2025-12-05T16:01:05.765Z
2,KYD1241,"Folk Icon ""Jesus Christ ""","Folk Icon ""Jesus Christ "", Ukraine, Western Po...",https://krovets.ua/disk/products/LMYFJXHaqKTZQ...,unknown,Western Podillia - ethnographical region | pai...,painting,"gouache paints, cardboard",Online Museum of the traditional art of Ukrain...,MUSEU,http://creativecommons.org/licenses/by-nc-nd/4.0/,20th century,"Ternopil obl., Zbarazkyi rn., v. Chornyi lis",/1413/KYD1241,1413_KROVETS_Museum,Ukraine,1413_KROVETS_Museum,2025-12-05T16:01:05.765Z,2025-12-05T16:01:05.765Z
3,KYD1242,"Folk Icon ""The Virgin Mary""","Folk Icon ""The Virgin Mary"", Ukraine, Western ...",https://krovets.ua/disk/products/JQhNZnEgo1HYe...,unknown,Western Podillia - ethnographical region | pai...,painting,"gouache paints, cardboard",Online Museum of the traditional art of Ukrain...,MUSEU,http://creativecommons.org/licenses/by-nc-nd/4.0/,20th century,"Ternopil obl., Zbarazkyi rn., v. Chornyi lis",/1413/KYD1242,1413_KROVETS_Museum,Ukraine,1413_KROVETS_Museum,2025-12-05T16:01:05.765Z,2025-12-05T16:01:05.765Z
4,KYD1243,"Folk Icon ""The Christ""","Folk Icon ""The Christ"", Ukraine, Poltavshchyna...",https://krovets.ua/disk/products/ZTEI5hDDqRPcK...,unknown,Poltavshchyna - ethnographical region | Wood |...,painting,"oil paints, wood",Online Museum of the traditional art of Ukrain...,MUSEU,http://creativecommons.org/licenses/by-nc-nd/4.0/,20th century,"Polatava obl., Hadiatskyi rn.",/1413/KYD1243,1413_KROVETS_Museum,Ukraine,1413_KROVETS_Museum,2025-12-05T16:01:05.765Z,2025-12-05T16:01:05.765Z


## Save to CSV

Export the DataFrame to `krovets_folk_metadata.csv` for use in downstream notebooks.

In [ ]:
# Update this path to your own Drive/local project folder before running.
OUTPUT_CSV = "/content/drive/MyDrive/W2L/Projects/AISTER/w2l/step_1/outputs/krovets_folk_metadata.csv"

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved {len(df)} rows to krovets_folk_metadata.csv")

Saved 312 rows to krovets_metadata.csv
